In [154]:
import pandas as pd
import re

In [125]:
dataset = pd.read_csv(r"/content/sentiment_analysis.csv")
dataset.head(3)

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook


In [126]:
dataset.isnull().sum()

,0
Year,0
Month,0
Day,0
Time of Tweet,0
text,0
sentiment,0
Platform,0


In [127]:
dataset.drop(columns = ["Year", "Month", "Day"], inplace =  True, axis = 1)

In [128]:
dataset

,Time of Tweet,text,sentiment,Platform
0,morning,What a great day!!! Looks like dream.,positive,Twitter
1,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,night,Don't angry me,negative,Facebook
3,morning,We attend in the class just for listening teac...,negative,Facebook
4,noon,"Those who want to go, let them go",negative,Instagram
...,...,...,...,...
494,night,"According to , a quarter of families under six...",negative,Twitter
495,morning,the plan to not spend money is not going well,negative,Instagram
496,noon,uploading all my bamboozle pictures of facebook,neutral,Facebook
497,night,congratulations ! you guys finish a month ear...,positive,Twitter


In [129]:
# dataset["Time of Tweet"].nunique()
dataset["sentiment"].unique()
dataset["Platform"].unique()


array([' Twitter  ', ' Facebook ', 'Facebook', ' Instagram ', ' Twitter '],
      dtype=object)

In [130]:
def make_changes(platform):
    if platform == " Facebook ":
        return "Facebook"
    elif platform == ' Twitter  ' or platform == ' Twitter ':
        return "Twitter"
    else:
        return "Instagram"
    # array([' Twitter  ', 'Facebook', ' Instagram ', 'Twitter'], dtype=object)

In [131]:
dataset["Platform"] = dataset["Platform"].apply(make_changes)

In [132]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Time of Tweet  499 non-null    object
 1   text           499 non-null    object
 2   sentiment      499 non-null    object
 3   Platform       499 non-null    object
dtypes: object(4)
memory usage: 15.7+ KB


In [133]:
cat = ["Time of Tweet", "Platform"]

In [134]:
X = dataset.drop("sentiment", axis = 1)
y = dataset["sentiment"]

In [135]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [136]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
# cat
# ohe = OneHotEncoder(drop = "first")

# for col in cat:
    # X_train[col] =  ohe.fit_transform(X_train[col], X_test[col])

preprocessor = ColumnTransformer(transformers = [("ohe", OneHotEncoder(handle_unknown = "ignore"), cat)])

preprocessor.fit(X_train)
X_train_enc =  preprocessor.transform(X_train)
X_test_enc = preprocessor.transform(X_test)



In [137]:
X_train_df = pd.DataFrame(X_train_enc, columns = preprocessor.get_feature_names_out())
X_test_df = pd.DataFrame(X_test_enc, columns =  preprocessor.get_feature_names_out())


In [138]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
X_train_df = X_train_df.reset_index(drop=True)
X_test_df = X_test_df.reset_index(drop=True)

X_train.drop(columns = cat, axis = 1, inplace = True)
X_test.drop(columns = cat, axis = 1, inplace = True)

X_train = pd.concat([X_train, X_train_df], axis = 1)
X_test = pd.concat([X_test, X_test_df], axis = 1)

,0
text,0
ohe__Time of Tweet_morning,0
ohe__Time of Tweet_night,0
ohe__Time of Tweet_noon,0
ohe__Platform_Facebook,0
ohe__Platform_Instagram,0
ohe__Platform_Twitter,0


In [142]:
# ("scaler", StandardScaler(), cat )

In [143]:
!pip install gensim

In [144]:
from gensim.models import Word2Vec

In [145]:
X_train_tokens = X_train["text"].str.split("")

In [155]:
def clean(text):
  text = str(text)
  lower = text.lower()
  text = re.sub(r"[^a-zA-Z ]", "", text)
  return lower.split()


In [156]:
train_token =  X_train["text"].apply(clean)
test_token =  X_test["text"].apply(clean)

In [157]:
train_token

,text
0,"[i, have, bad, headech,, what, i, need, to, do..."
1,"[happy, mother`s, day, to, all, the, mothers, ..."
2,"[getting, ready, for, week, its, too, nice, to..."
3,"[back, soon,, need, to, run, to, the, shops, a..."
4,"[errors, are, red, but, my, life, is, blue,, i..."
...,...
394,"[car, not, happy,, big, big, dent, in, boot!, ..."
395,"[him, shirt, at, dinner?, do, you, need, to, a..."
396,"[i, only, do, computers., am, hopeless, at, ev..."
397,"[no, waterfront, anymore, faccia, luna, and, c..."


In [159]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
